## 1. Kiểm tra môi trường (PyTorch & CUDA)

In [ ]:
import torch
print(torch.__version__)        # cần >= 2.7.0
print(torch.cuda.is_available()) # phải là True

## 2. Cài đặt dependencies

In [ ]:
!git clone https://github.com/facebookresearch/sam3.git
%cd sam3
!pip install . -q

In [ ]:
from huggingface_hub import login
login(token="YOUR_HUGGINGFACE_TOKEN") # thay YOUR_HUGGING_FACE_TOKEN bằng token của bạn

In [ ]:
!pip install "numpy==1.26.4" -q

import os
os.kill(os.getpid(), 9)

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

In [ ]:
import torch
import torchvision
import numpy as np

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("numpy:", np.__version__)
print("CUDA:", torch.cuda.is_available())

In [ ]:
# Đường dẫn tới ảnh cần test
IMAGE_PATH = "/path/to/your/test_image.jpg"  # Thay bằng đường dẫn thực tế tới ảnh của bạn

# Prompt chính dùng để visualize bounding box + mask
MAIN_PROMPT = "a photo of a cat"  # Thay bằng prompt phù hợp với ảnh của bạn

# Đường dẫn lưu ảnh kết quả
OUTPUT_PATH = "sam3_result.png"
# ============================================================

# Chạy inference với model được fine-tune

In [ ]:
%cd /kaggle/working/sam3

import torch
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# Load config từ cell đầu (nhớ bỏ comment để dùng nếu bạn đã set biến ở trên)
# %store -r IMAGE_PATH MAIN_PROMPT COMPARE_PROMPTS OUTPUT_PATH

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")

# 1. Khởi tạo kiến trúc mô hình gốc
BPE_PATH = "/kaggle/working/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
model = build_sam3_image_model(bpe_path=BPE_PATH)

# 2. Đọc và nạp file checkpoint bạn vừa huấn luyện
CHECKPOINT_PATH = "/path/to/your/sam3_checkpoint.pth"  # Thay bằng đường dẫn thực tế tới checkpoint của bạn
print(f"Đang nạp trọng số từ: {CHECKPOINT_PATH}")

# Load file weights (lên CPU trước để tránh sốc VRAM)
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)

# Lấy phần dictionary chứa trọng số (do lớp Trainer của SAM3 thường bọc trong key 'model')
state_dict = checkpoint['model'] if 'model' in checkpoint else checkpoint

# Bơm trọng số vào kiến trúc mạng
model.load_state_dict(state_dict, strict=False)

# Đẩy mô hình lên GPU và khóa đạo hàm (Eval mode)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# 3. Khởi tạo Processor 
processor = Sam3Processor(model)
print("✅ Nạp thành công Custom Model!")

# =======================================================
# 4. CHẠY DỰ ĐOÁN (INFERENCE)
# =======================================================
image = Image.open(IMAGE_PATH).convert("RGB")
print(f"Ảnh: {IMAGE_PATH} | Size: {image.size}")

# Đưa ảnh vào bộ nhớ đệm của processor
inference_state = processor.set_image(image)

# Đưa prompt dạng text (Ví dụ: "aquarium", "fish", "tank"...) vào để nhận diện
output = processor.set_text_prompt(state=inference_state, prompt=MAIN_PROMPT)

masks, boxes, scores = output["masks"], output["boxes"], output["scores"]

# In kết quả của vật thể có điểm tin cậy (score) cao nhất
if len(scores) > 0:
    print(f"Prompt: '{MAIN_PROMPT}' | Điểm tự tin cao nhất (Score): {scores[0].item():.4f}")
    print(f"Tọa độ Bounding Box: {boxes[0].tolist()}")
else:
    print(f"Không tìm thấy '{MAIN_PROMPT}' trong ảnh này.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Ảnh gốc + bounding box
axes[0].imshow(image)
axes[0].set_title(f"Bounding Box — '{MAIN_PROMPT}'")
for box in boxes:
    x1, y1, x2, y2 = box.cpu().numpy()
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                               linewidth=2, edgecolor='red', facecolor='none')
    axes[0].add_patch(rect)
axes[0].axis('off')

# Ảnh gốc + mask
axes[1].imshow(image)
axes[1].set_title(f"Segmentation Mask (score: {scores[0].item():.4f})")
mask = masks[0].cpu().numpy().squeeze()
axes[1].imshow(mask, alpha=0.5, cmap='jet')
axes[1].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f"Đã lưu kết quả vào {OUTPUT_PATH}")